In [0]:
#nationality-country,nationalityregion-continent

## Build Constructors Dimension
- 1.Build silver constructore dimension
- 2.Build gold ref_nationality_region table
- 3.Join the data from constructors with ref_nationality_region using nationality
- 4.select the required columns
- 5.Write transformed data to gold dim_constructors table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/04.gold-help

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
import pyspark.sql.functions as F

**Step -1 Read source tables**
- 
- silver.constructors
- gold.ref_nationality_region

In [0]:
constructors_df=(
    spark.table(f"{catalog_name}.{silver_schema}.constructors").filter(F.col("batch_id") == v_batch_id)
)

In [0]:

ref_nationality_region_df=spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

**Step 2 Join the contructors with ref_nationality_region using nationality**

In [0]:
dim_constructors_df=(
    constructors_df
    .join(
        ref_nationality_region_df,
        constructors_df.nationality==ref_nationality_region_df.nationality,
        "left"
    )
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        ref_nationality_region_df.region.alias('nationality_region')
        
    )
)

In [0]:
display(dim_constructors_df)

constructor_id,constructor_name,nationality,nationality_region
ats,ATS,Italian,Europe
benetton,Benetton,Italian,Europe
bmw,BMW,German,Europe
brabham-repco,Brabham-Repco,British,Europe
cadillac,Cadillac F1 Team,American,North America
force_india,Force India,Indian,Asia
lotus-pw,Lotus-Pratt & Whitney,British,Europe
osella,Osella,Italian,Europe
token,Token,British,Europe
amon,Amon,New Zealander,Oceania


In [0]:
write_to_gold(
    dim_constructors_df,
    target_table,
    merge_condition="s.constructor_id=t.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))

constructor_id,constructor_name,nationality,nationality_region,created_timestamp,updated_timestamp
ats,ATS,Italian,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
benetton,Benetton,Italian,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
bmw,BMW,German,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
brabham-repco,Brabham-Repco,British,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
cadillac,Cadillac F1 Team,American,North America,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
force_india,Force India,Indian,Asia,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
lotus-pw,Lotus-Pratt & Whitney,British,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
osella,Osella,Italian,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
token,Token,British,Europe,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
amon,Amon,New Zealander,Oceania,2026-10-20T12:20:26.940Z,2026-10-22T11:03:21.782Z
